<a href="https://colab.research.google.com/github/umetaisa2004-commits/content-ops-starter/blob/main/20261003DataManagement_to_Origin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import lognorm
from google.colab import files
import pandas as pd

# ==========================================
# 粒度分布の計算とOrigin用レイアウト整理
# ==========================================

# 1. 描画したい時間を指定
target_times = [10.0, 100.0, 200.0]

# 2. X軸（粒径）の設定
x_diameters = np.linspace(1e-6, 2000e-6, 500)
dx = x_diameters[1] - x_diameters[0]

# 列を分けて保存するための辞書を準備
pdf_cols = {'Diameter_m': x_diameters}
cdf_cols = {'Diameter_m_CDF': x_diameters} # Origin用の2つ目のX軸

plt.figure(figsize=(10, 6))

for t in target_times:
    idx = (merged_df['Flow_Time'] - t).abs().idxmin()
    row = merged_df.loc[idx]
    actual_time = row['Flow_Time']

    D43 = row['D43']
    variance = row['variance']

    if D43 > 0:
        sigma_ln_sq = np.log(1 + (variance / (D43**2)))
        sigma_ln = np.sqrt(sigma_ln_sq)
        mu_ln = np.log(D43) - (sigma_ln_sq / 2)

        # ① 頻度分布（PDF）
        pdf_values = lognorm.pdf(x_diameters, s=sigma_ln, scale=np.exp(mu_ln))
        true_prob_percent = pdf_values * dx * 100
        pdf_cols[f'PDF_{actual_time}s(%)'] = true_prob_percent

        # ② 累積分布（CDF）
        cdf_percent = lognorm.cdf(x_diameters, s=sigma_ln, scale=np.exp(mu_ln)) * 100
        cdf_cols[f'Cumulative_{actual_time}s(%)'] = cdf_percent

        plt.plot(x_diameters * 1e6, true_prob_percent, lw=2,
                 label=f'{actual_time} s (D43={D43*1e6:.0f} μm)')

plt.title('Particle Size Distribution (Frequency %)', fontsize=14)
plt.xlabel('Diameter (μm)', fontsize=12)
plt.ylabel(f'Frequency (%) per {dx*1e6:.1f} μm bin', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.xlim(0, 2000)
plt.tight_layout()
plt.show()

# ==========================================
# 3. 指定の列順で結合して保存
# ==========================================
# 頻度のデータフレームと累積のデータフレームを横に結合
df_pdf = pd.DataFrame(pdf_cols)
df_cdf = pd.DataFrame(cdf_cols)
final_out_df = pd.concat([df_pdf, df_cdf], axis=1)

csv_filename = 'Origin_Arranged_Distributions.csv'
final_out_df.to_csv(csv_filename, index=False)
print(f"Origin用データ（指定レイアウト）を保存しました: {csv_filename}")
files.download(csv_filename)

KeyError: 'Flow_Time'

<Figure size 1000x600 with 0 Axes>

In [ ]:
import pandas as pd
import glob
from google.colab import files

file_list = glob.glob('*.out')
merged_df = pd.DataFrame()

for i, file in enumerate(file_list):
    print(f"【読み込み中】: {file}")

    param_name = file.replace('report-def-', '').replace('-rfile.out', '')

    df = pd.read_csv(
        file,
        sep=r'\s+',  # 正規表現で空白区切りを指定
        skiprows=3,
        names=['Time_Step', param_name, 'Flow_Time']
    )

    if i == 0:
        merged_df = df[['Time_Step', 'Flow_Time', param_name]]
    else:
        df_subset = df[['Time_Step', param_name]]
        merged_df = pd.merge(merged_df, df_subset, on='Time_Step', how='outer')

# ==========================================
# 🌟 ここからが列を綺麗に並べ替える追加処理です
# ==========================================

# 1. 全ての列名を取得
all_columns = merged_df.columns.tolist()

# 2. 'Time_Step' と 'Flow_Time' 以外の列（ab, m, wなど）だけを抽出
param_columns = [col for col in all_columns if col not in ['Time_Step', 'Flow_Time']]

# 3. パラメータの列をアルファベット順＆数字順に並べ替え
param_columns_sorted = sorted(param_columns)

# 4. 「固定の2列」＋「綺麗に並べ替えたパラメータ列」を結合
final_column_order = ['Time_Step', 'Flow_Time'] + param_columns_sorted
merged_df = merged_df[final_column_order]

# ==========================================
# 🌟 【新規追加】D43 と variance の計算列を追加
# ==========================================
# m3, m4, m5 のモーメント値から体積平均径(D43)と分散(variance)を計算します。
# ※もし計算式が異なる場合（例えば QMOMの ab と w から直接計算する場合）は適宜書き換えてください。

if 'm3' in merged_df.columns and 'm4' in merged_df.columns:
    # 体積平均径 (D43) = m4 / m3
    merged_df['D43'] = merged_df['m4'] / merged_df['m3']

    if 'm5' in merged_df.columns:
        # 体積基準の分散 = (m5 / m3) - (m4 / m3)^2
        merged_df['variance'] = (merged_df['m5'] / merged_df['m3']) - (merged_df['D43'] ** 2)
    else:
        print("※警告: 'm5' 列が存在しないため 'variance' は計算されませんでした。")
else:
    print("※警告: 'm3' または 'm4' 列が存在しないため 'D43' は計算されませんでした。")

# (おまけ) ザウター平均径 (D32) も計算しておく場合
if 'm2' in merged_df.columns and 'm3' in merged_df.columns:
    merged_df['D32'] = merged_df['m3'] / merged_df['m2']

# ==========================================

# 行をTime_Step順に並び替え
merged_df = merged_df.sort_values('Time_Step')

# 1つのExcelファイルとして保存
output_filename = 'All_Parameters_Merged.xlsx'
merged_df.to_excel(output_filename, index=False)

print(f"\n【処理完了】列を整理し、D43等の計算列を追加しました！ {output_filename} をダウンロードします...")
files.download(output_filename)

【読み込み中】: report-def-m5-rfile.out
【読み込み中】: report-def-w2-rfile.out
【読み込み中】: report-def-m3-rfile.out
【読み込み中】: report-def-w0-rfile.out
【読み込み中】: report-def-ab2-rfile.out
【読み込み中】: report-def-m1-rfile.out
【読み込み中】: report-def-ab0-rfile.out
【読み込み中】: report-def-m0-rfile.out
【読み込み中】: report-def-m4-rfile.out
【読み込み中】: report-def-w1-rfile.out
【読み込み中】: report-def-ab1-rfile.out
【読み込み中】: report-def-m2-rfile.out

【処理完了】列を整理し、D43等の計算列を追加しました！ All_Parameters_Merged.xlsx をダウンロードします...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>